# CAMUS — Test-Set Evaluation

Evaluates the trained UNet (ResNet34) checkpoints on the **held-out test set** without retraining.

The 80/10/10 split is reproduced exactly as in `camus_segmentation.py`: the same sorted file list and
`train_test_split(..., random_state=42)` give the same test patients the model never saw.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

In [ ]:
!pip install -q segmentation-models-pytorch SimpleITK

In [ ]:
import os
from glob import glob

import cv2
import numpy as np
import SimpleITK as sitk
import skimage.transform
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Same locations used during training
DATA_PATH = "/content/drive/MyDrive/ch_database"
CHECKPOINTS = {
    "best_model (selected by validation loss)": "/content/drive/MyDrive/best_model.pth",
    "final_model (last epoch)": "/content/drive/MyDrive/final_model.pth",
}
NUM_CLASSES = 4
CLASS_NAMES = ["Background", "LV Endocardium", "LV Myocardium", "Left Atrium"]

## Data — identical preprocessing and split to training

In [ ]:
def preprocess(path, is_mask=False, target_size=(320, 320)):
    """Same preprocessing as camus_segmentation.py."""
    data = sitk.GetArrayFromImage(sitk.ReadImage(path, sitk.sitkFloat32))
    if data.ndim == 3:
        data = data[0]  # first slice

    if is_mask:
        data_resized = skimage.transform.resize(
            data, target_size, order=0, preserve_range=True, anti_aliasing=False
        ).astype(np.uint8)
        return torch.from_numpy(data_resized).long()

    data_resized = cv2.resize(data, target_size, interpolation=cv2.INTER_CUBIC)
    data_resized = data_resized / 255.0
    return torch.from_numpy(np.expand_dims(data_resized, axis=0)).float()


class CHDataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths = image_paths
        self.mask_paths = mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        return (preprocess(self.image_paths[idx], is_mask=False),
                preprocess(self.mask_paths[idx], is_mask=True))


def get_image_mask_pairs(data_path):
    """Same file ordering as training (sorted glob per view)."""
    image_paths, mask_paths = [], []
    for view in ['2CH', '4CH']:
        image_dir = os.path.join(data_path, view, 'images')
        mask_dir = os.path.join(data_path, view, 'masks')
        for img_path in sorted(glob(os.path.join(image_dir, '*.nii.gz'))):
            filename = os.path.basename(img_path)
            mask_path = os.path.join(mask_dir, filename.replace('.nii.gz', '_gt.nii.gz'))
            if os.path.exists(mask_path):
                image_paths.append(img_path)
                mask_paths.append(mask_path)
    return image_paths, mask_paths


image_paths, mask_paths = get_image_mask_pairs(DATA_PATH)

# Reproduce the 80/10/10 split with the same random_state
img_train, img_rem, msk_train, msk_rem = train_test_split(
    image_paths, mask_paths, train_size=0.8, random_state=42)
img_val, img_test, msk_val, msk_test = train_test_split(
    img_rem, msk_rem, test_size=0.5, random_state=42)

print(f"Total pairs: {len(image_paths)} | train {len(img_train)} | val {len(img_val)} | test {len(img_test)}")

test_loader = DataLoader(CHDataset(img_test, msk_test), batch_size=32, shuffle=False,
                         num_workers=2, pin_memory=True)

## Metrics — same definitions as training

In [ ]:
def per_class_dice(preds, targets, num_classes):
    preds, targets = preds.view(-1), targets.view(-1)
    out = []
    for c in range(num_classes):
        p, t = preds == c, targets == c
        out.append((2. * (p & t).sum().item()) / (p.sum().item() + t.sum().item() + 1e-8))
    return out


def per_class_iou(preds, targets, num_classes):
    preds, targets = preds.view(-1), targets.view(-1)
    out = []
    for c in range(num_classes):
        p, t = preds == c, targets == c
        union = (p | t).sum().item()
        out.append(np.nan if union == 0 else (p & t).sum().item() / union)
    return out


def per_class_accuracy(preds, targets, num_classes):
    preds, targets = preds.view(-1), targets.view(-1)
    out = []
    for c in range(num_classes):
        p, t = preds == c, targets == c
        total = t.sum().item()
        out.append(np.nan if total == 0 else (p & t).sum().item() / total)
    return out


def macro(values):
    values = [v for v in values if not np.isnan(v)]
    return float(np.mean(values)) if values else float("nan")

## Evaluate each checkpoint on the test set

In [ ]:
criterion = nn.CrossEntropyLoss()
results = {}

for label, ckpt_path in CHECKPOINTS.items():
    if not os.path.exists(ckpt_path):
        print(f"⚠️  Not found, skipping: {ckpt_path}")
        continue

    model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=1, classes=NUM_CLASSES)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.to(device).eval()

    running_loss, all_preds, all_targets = 0.0, [], []
    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc=label):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            running_loss += criterion(outputs, masks).item()
            all_preds.append(torch.argmax(outputs, dim=1).cpu())
            all_targets.append(masks.cpu())

    preds, targets = torch.cat(all_preds), torch.cat(all_targets)
    dice = per_class_dice(preds, targets, NUM_CLASSES)
    iou = per_class_iou(preds, targets, NUM_CLASSES)
    acc = per_class_accuracy(preds, targets, NUM_CLASSES)

    results[label] = {
        "loss": running_loss / len(test_loader),
        "dice": macro(dice), "iou": macro(iou), "acc": macro(acc),
        "dice_fg": macro(dice[1:]),  # 3 cardiac structures only (no background)
        "dice_per_class": dice, "iou_per_class": iou,
    }

    print(f"\n=== {label} ===")
    print(f"Test loss: {results[label]['loss']:.4f}")
    print(f"Dice (macro, 4 classes): {results[label]['dice']:.4f}")
    print(f"Dice (3 cardiac structures, no background): {results[label]['dice_fg']:.4f}")
    print(f"IoU (macro): {results[label]['iou']:.4f} | Accuracy (macro): {results[label]['acc']:.4f}")
    for name, d, j in zip(CLASS_NAMES, dice, iou):
        print(f"  {name:<15} Dice {d:.4f} | IoU {j:.4f}")

## Summary — copy this output and paste it back

In [ ]:
print("| Checkpoint | Test Loss | Dice (4 classes) | Dice (cardiac structures) | IoU | Accuracy |")
print("|---|---|---|---|---|---|")
for label, r in results.items():
    print(f"| {label} | {r['loss']:.4f} | {r['dice']:.4f} | {r['dice_fg']:.4f} | {r['iou']:.4f} | {r['acc']:.4f} |")

print("\nPer-class test Dice:")
print("| Checkpoint | " + " | ".join(CLASS_NAMES) + " |")
print("|---|" + "---|" * len(CLASS_NAMES))
for label, r in results.items():
    print(f"| {label} | " + " | ".join(f"{d:.4f}" for d in r["dice_per_class"]) + " |")